# Notebook 014: Spark Higher-Order Functions: Transform, Filter and Aggregate

**Module:** Module 02: Complex Data Types & Nested Structures Handling  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 12: Debugging Code in Databricks & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
A fintech loyalty platform tracks customer reward points earned across multiple merchant transactions in an array. The team needs to adjust point values, filter out expired promo points, and sum net balances without exploding rows.

## 2. Technical Requirement & Problem Statement
Implement Spark 3+ Higher-Order Functions directly on arrays: `transform` (lambda mapping), `filter` (predicate filtering), `aggregate` (reduce/fold), `exists`, and `forall` without incurring expensive explode-and-groupBy shuffles.

## 3. Spark & Databricks Architecture Notes
Higher-order functions run inline lambda expressions inside the JVM on array elements without flattening the DataFrame. Exploding an array multiplies the row count and requires a subsequent `groupBy()` shuffle to reconstitute the row, causing 10x-100x slower execution.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_014_Spark_Higher-Order_Functions_Transform,_Filter_and_Aggregate")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating Loyalty Transactions Data
Create accounts with arrays of point earnings.

In [ ]:
loyalty_data = [
    ("ACC-1", [150, 45, 800, 20, 310]),
    ("ACC-2", [50, 25, 10]),
    ("ACC-3", [1200, 450, 90, 850]),
    ("ACC-4", [5, 12, 18])
]

df_loyalty = spark.createDataFrame(loyalty_data, ["account_id", "point_events"])
df_loyalty.show(truncate=False)

## 6. Transform & Filter Higher-Order Functions
Apply 10% bonus to points over 100 and filter out micro-transactions (< 30 points).

In [ ]:
# 1. Transform: add 10% bonus to all points: x -> cast(x * 1.10 as int)
df_transformed = df_loyalty.withColumn(
    "bonus_points",
    F.expr("transform(point_events, x -> cast(x * 1.10 AS INT))")
)

# 2. Filter: keep only significant transactions >= 50
df_filtered = df_transformed.withColumn(
    "significant_events",
    F.expr("filter(point_events, x -> x >= 50)")
)

df_filtered.show(truncate=False)

## 7. Aggregate (Reduce), Exists and Forall
Compute running sum via aggregate and evaluate boolean array conditions.

In [ ]:
# Aggregate: sum all points starting from initial accumulator 0: (acc, x) -> acc + x
df_agg = df_filtered.withColumn(
    "total_points",
    F.expr("aggregate(point_events, 0, (acc, x) -> acc + x)")
).withColumn(
    "has_super_bonus",
    F.expr("exists(point_events, x -> x >= 500)")
).withColumn(
    "all_positive",
    F.expr("forall(point_events, x -> x > 0)")
)

df_agg.select("account_id", "point_events", "total_points", "has_super_bonus", "all_positive").show(truncate=False)

## 9. Key Architectural Takeaways
- **Higher-order functions eliminate the need for `explode()` and `groupBy()` when manipulating array elements.**
- **`transform` maps every element to a new value via lambda expression.**
- **`aggregate(arr, zero, (acc, x) -> ...)` performs in-memory reduction without network shuffle.**

## 10. Senior Data Engineering Interview Questions
1. Why are Higher-Order Functions significantly faster than `explode` followed by `groupBy`?
2. How does `exists(arr, x -> cond)` differ from `array_contains(arr, val)`?
3. Can you use custom Python functions inside Spark's `transform` higher-order function?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
